# S&P 500 Valuation Analysis Pipeline 📈

**ECM Research | Bulge Bracket Bank**

This notebook automates the extraction, cleaning, and analysis of S&P 500 constituent data.
It scrapes live metrics from **Finviz** and historical P/E benchmarks from **Multpl.com**,
then applies **three Claude-powered Gen AI augmentations** plus an **Agentic RAG** and **Graph RAG** layer.

### Pipeline Architecture:
| Step | Component | Type |
|------|-----------|------|
| 1 | Historical P/E Benchmark (Multpl.com) | Deterministic |
| 2 | Live Finviz Screener Scraper | Deterministic |
| 3 | Data Cleaning & Standardisation | Deterministic |
| 4 | Sector Aggregations | Deterministic |
| 5 | Excel Dashboard Builder | Deterministic |
| **6** | **Claude API Setup** | **✨ Gen AI** |
| **7** | **AI Executive Commentary** | **✨ Gen AI** |
| **8** | **Natural Language Query (NLQ)** | **✨ Gen AI** |
| **9** | **AI Qualitative Risk Flags** | **✨ Gen AI** |
| **10** | **Agentic RAG** | **✨ Gen AI** |
| **11** | **Graph RAG** | **✨ Gen AI** |
| 12 | Unit Tests | Programmatic |

### Setup:
Create a `.env` file in the same folder as this notebook:
```
ANTHROPIC_API_KEY=sk-ant-your-key-here
```
Get your key at [console.anthropic.com](https://console.anthropic.com)


## Step 1: Scrape Dynamic Historical Benchmark
Instead of hardcoding a 17.5x P/E ratio, we scrape **Multpl.com** for over 150 years of Trailing Twelve Month (TTM) P/E data. 

*Note: We calculate the **Median** rather than the Mean. The Mean is heavily skewed by massive spikes during recessions (e.g., the 2009 crash where P/E mathematically hit 123x as earnings approached zero). The median provides a much more accurate "normal" historical baseline.*


In [1]:
import argparse
import re
import sys
import json
import time
import random
import logging
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────────────────────
# LOGGING SETUP
# ─────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("sp500")

# ─────────────────────────────────────────────────────────────
# GLOBAL SESSION & HEADERS
# ─────────────────────────────────────────────────────────────
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
})

# ─────────────────────────────────────────────────────────────
# HISTORICAL S&P 500 P/E BENCHMARK (MULTPL.COM)
# ─────────────────────────────────────────────────────────────
MULTPL_PE_URL = "https://www.multpl.com/s-p-500-pe-ratio/table/by-year"
FALLBACK_HIST_PE   = 17.5

def scrape_sp500_historical_pe() -> float:
    """Scrapes historical S&P 500 P/E from Multpl.com and returns the median value."""
    try:
        headers = session.headers.copy()
        resp = session.get(MULTPL_PE_URL, headers=headers, timeout=15)
        resp.raise_for_status()
        
        # Regex to find all table rows with Date and Value cells safely
        rows = re.findall(r'<tr.*?<td>(.*?)</td>.*?<td>(.*?)</td', resp.text, re.DOTALL)
        
        pe_values = []
        for date_str, val_str in rows:
            if "Date" in date_str or "Value" in date_str:
                continue
                
            # Clean HTML tags, estimate crosses (†), and unicode spaces
            val_clean = re.sub(r'<[^>]+>', '', val_str).strip()
            val_clean = val_clean.replace('†', '').replace('&#x2002;', '').strip()
            val_clean = re.sub(r'[^\d.]', '', val_clean)
            
            if val_clean:
                pe_values.append(float(val_clean))
        
        if not pe_values:
            raise ValueError("No valid numeric P/E values extracted.")
            
        # Calculate the median manually
        pe_values.sort()
        n = len(pe_values)
        if n % 2 == 0:
            median_pe = (pe_values[n//2 - 1] + pe_values[n//2]) / 2.0
        else:
            median_pe = pe_values[n//2]
            
        return round(float(median_pe), 2)

    except Exception as e:
        log.warning(f"Multpl.com scrape failed: {str(e)}. Defaulting to {FALLBACK_HIST_PE}x.")
        return FALLBACK_HIST_PE
  



# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────
# 1. Fetch historical average safely
HIST_AVG_PE = scrape_sp500_historical_pe()
if HIST_AVG_PE != FALLBACK_HIST_PE:
    log.info(f"Successfully scraped Multpl.com. Historical Median P/E: {HIST_AVG_PE}x")

# 2. Establish High/Low thresholds
PE_HIGH_BOUND = HIST_AVG_PE + 2.5
PE_LOW_BOUND  = HIST_AVG_PE - 2.5

# 3. Finviz Settings
FINVIZ_BASE   = "https://finviz.com/screener.ashx"
FINVIZ_COLS   = "0,1,2,3,4,6,7,8,14,16,77,67,65,66"
FINVIZ_FILTER = "idx_sp500"
PAGE_SIZE     = 20


20:26:56  INFO     Successfully scraped Multpl.com. Historical Median P/E: 15.22x


## Step 2: Live Finviz Screener Scraper
This class connects to Finviz, identifies the total number of companies in the S&P 500, and auto-paginates through the results until all constituent data is downloaded.


In [2]:
# ─────────────────────────────────────────────────────────────
# STEP 1 — FINVIZ SCRAPER
# ─────────────────────────────────────────────────────────────

class FinvizScraper:
    """
    Auto-paginating Finviz screener scraper.
    - Discovers total result count on first page
    - Iterates r=1, r=21, r=41 … until all rows fetched
    - Retries with exponential back-off on network errors
    - Flexible column parsing: reads whatever headers Finviz returns
    """

    def __init__(self, delay: float = 1.2, max_retries: int = 3):
        self.delay = delay
        self.max_retries = max_retries
        self.session = session

    # ── HTTP ──────────────────────────────────────────────────

    def _get(self, params: dict) -> BeautifulSoup:
        for attempt in range(1, self.max_retries + 1):
            try:
                resp = self.session.get(FINVIZ_BASE, params=params, timeout=20)
                resp.raise_for_status()
                return BeautifulSoup(resp.text, "lxml")
            except requests.RequestException as exc:
                wait = attempt * 3 + random.uniform(0, 2)
                log.warning(
                    f"Attempt {attempt}/{self.max_retries} failed ({exc}). "
                    f"Retrying in {wait:.1f}s …"
                )
                time.sleep(wait)
        raise RuntimeError(
            f"Finviz unreachable after {self.max_retries} attempts. "
            "Check network or run with --demo flag for offline testing."
        )

    # ── Parsing ──────────────────────────────────────────────

    @staticmethod
    def _total_count(soup: BeautifulSoup) -> int:
        """Extract 'Total: NNN' from page. Returns 0 if not found."""
        for text in soup.stripped_strings:
            m = re.search(r"Total:\s*(\d+)", text)
            if m:
                return int(m.group(1))
        return 0

    @staticmethod
    def _find_table(soup: BeautifulSoup):
        """Locate the main screener results table robustly."""
        # Try known class names first
        for selector in [
            {"class": "screener_table"},
            {"id": "screener-views-table"},
        ]:
            t = soup.find("table", selector)
            if t:
                return t
        # Fallback: any table that contains screener-link-primary anchors
        for t in soup.find_all("table"):
            if t.find("a", class_="screener-link-primary"):
                return t
        return None

    def _parse_page(self, soup: BeautifulSoup) -> list[dict]:
        table = self._find_table(soup)
        if not table:
            log.warning("Screener table not found on this page — layout may have changed.")
            return []

        all_rows = table.find_all("tr")
        if not all_rows:
            return []

        # First row = headers
        headers = [th.get_text(strip=True) for th in all_rows[0].find_all(["th", "td"])]

        rows = []
        for tr in all_rows[1:]:
            cells = tr.find_all("td")
            if not cells:
                continue
            row = {}
            for i, td in enumerate(cells):
                key = headers[i] if i < len(headers) else f"col_{i}"
                a = td.find("a", class_="screener-link-primary")
                row[key] = a.get_text(strip=True) if a else td.get_text(strip=True)
            # keep only real stock rows (Ticker looks like a ticker)
            ticker_val = row.get("Ticker", row.get("No.", ""))
            if any(c.isalpha() for c in ticker_val):
                rows.append(row)

        return rows

    # ── Public API ──────────────────────────────────────────

    def fetch_sp500(self, max_rows: int | None = None) -> pd.DataFrame:
        log.info("Connecting to Finviz S&P 500 screener …")
        params = dict(v=152, f=FINVIZ_FILTER, ft=4, r=1, c=FINVIZ_COLS)

        # Page 1 — discover total
        soup  = self._get(params)
        total = self._total_count(soup)
        if total == 0:
            log.warning("Could not read total count — will paginate until empty page.")
            total = 600

        log.info(f"Finviz reports {total} S&P 500 constituents.")
        rows = self._parse_page(soup)
        log.info(f"  Page r=1 → {len(rows)} rows")
        time.sleep(self.delay + random.uniform(0, 0.4))

        # Remaining pages
        r = PAGE_SIZE + 1
        while r <= total:
            if max_rows and len(rows) >= max_rows:
                break
            params["r"] = r
            log.info(f"  Fetching rows {r}–{r + PAGE_SIZE - 1} …")
            soup  = self._get(params)
            page  = self._parse_page(soup)
            if not page:
                log.info("  Empty page — pagination complete.")
                break
            rows.extend(page)
            log.info(f"  Running total: {len(rows)} rows")
            r += PAGE_SIZE
            time.sleep(self.delay + random.uniform(0, 0.4))

        if max_rows:
            rows = rows[:max_rows]

        df = pd.DataFrame(rows)
        log.info(f"Scrape complete: {len(df)} rows, {len(df.columns)} columns.")
        log.info(f"Columns returned by Finviz: {list(df.columns)}")
        return df


## Step 3: Data Cleaning & Standardization
This section maps the messy HTML headers from Finviz to our internal taxonomy. It strips out strings (like "B" for Billions and "%" for percentages) and safely coerces them into usable floats for analysis. It also assigns our dynamic Valuation Flags.


In [3]:

# Map any Finviz header variant → internal column names
COLUMN_ALIASES: dict[str, str] = {
    "no.":              "row_no",
    "#":                "row_no",
    "ticker":           "Ticker",
    "company":          "Company Name",
    "sector":           "GICS Sector",
    "industry":         "GICS Sub-Industry",
    "market cap":       "Market Cap ($M)",      # col 6
    "p/e":              "LTM P/E",              # col 7
    "fwd p/e":          "NTM P/E",              # col 8
    "forward p/e":      "NTM P/E",
    "eps":              "EPS TTM",              # col 16 (EPS ttm)
    "eps (ttm)":        "EPS TTM",
    "eps next q":             "CQ+1 EPS Estimate",
    "eps estimate next quarter": "CQ+1 EPS Estimate",
    "eps est. next q":        "CQ+1 EPS Estimate",  # col 77
    "dividend %":       "Dividend Yield (%)",   # col 14
    "dividend":         "Dividend Yield (%)",
    "div yield":        "Dividend Yield (%)",
    # "price":            "Share Price",          # col 65
    # "volume":           "Volume",               # col 67
    # "change":           "Change (%)",           # col 66 (daily % change)
}


def _parse_market_cap(val) -> float | None:
    """'2.85T' → 2_850_000, '342.10B' → 342_100, '8.50M' → 8.50 (all in $M)."""
    if not val or str(val).strip() in ("-", "", "N/A", "nan"):
        return None
    s = str(val).strip().replace(",", "")
    m = re.match(r"^([\d.]+)([TBMKtbmk]?)$", s)
    if not m:
        return None
    num    = float(m.group(1))
    suffix = m.group(2).upper()
    return num * {"T": 1_000_000, "B": 1_000, "M": 1, "K": 0.001}.get(suffix, 1)

def _parse_pct(val) -> float | None:
    """'2.45%' or '2.45' → 2.45 numeric."""
    if not val or str(val).strip() in ("-", "", "N/A", "nan"):
        return None
    try:
        return float(re.sub(r"[%,\s]", "", str(val)))
    except ValueError:
        return None

def _parse_float(val) -> float | None:
    if not val or str(val).strip() in ("-", "", "N/A", "nan"):
        return None
    try:
        return float(str(val).replace(",", ""))
    except ValueError:
        return None

def standardise(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Rename Finviz columns → internal names, coerce numerics,
    derive computed columns, add Valuation Flag.
    """
    df = df_raw.copy()

    # 1. Rename: case-insensitive match
    rename_map = {}
    for col in df.columns:
        key = col.strip().lower()
        if key in COLUMN_ALIASES:
            rename_map[col] = COLUMN_ALIASES[key]
    df.rename(columns=rename_map, inplace=True)

    # 2. Ensure required columns exist
    for req in [
        "Ticker","Company Name","GICS Sector","GICS Sub-Industry",
        "Market Cap ($M)","LTM P/E","NTM P/E",
        "Dividend Yield (%)","EPS YoY","EPS QoQ"
    ]:
        if req not in df.columns:
            df[req] = None

    # 3. Numeric coercion
    # EPS TTM and CQ+1 EPS from new columns
    df["EPS TTM"]           = df["EPS TTM"].apply(_parse_float)
    df["CQ+1 EPS Estimate"] = df["CQ+1 EPS Estimate"].apply(_parse_float)
    # Market cap, P/E, Div Yield as before
    df["Market Cap ($M)"]    = df["Market Cap ($M)"].apply(_parse_market_cap)
    df["LTM P/E"]            = df["LTM P/E"].apply(_parse_float)
    df["NTM P/E"]            = df["NTM P/E"].apply(_parse_float)
    df["Dividend Yield (%)"] = df["Dividend Yield (%)"].apply(_parse_pct)

    # # Price, Volume, Change
    # df["Share Price"] = df["Share Price"].apply(_parse_float)
    # df["Volume"]      = df["Volume"].apply(_parse_float)
    # df["Change (%)"]  = df["Change (%)"].apply(_parse_pct)


    # 4. Zero / negative P/E → None (exclude from valuation averages)
    for col in ["LTM P/E","NTM P/E"]:
        df.loc[df[col].notna() & (df[col] <= 0), col] = None

    # 5. Derived metrics
    total_mc = df["Market Cap ($M)"].sum(skipna=True)
    df["% of S&P 500 Index"] = (df["Market Cap ($M)"] / total_mc * 100
                                if total_mc else None)

    df["LTM P/E vs Hist Avg (%)"] = df["LTM P/E"].apply(
        lambda x: (x - HIST_AVG_PE) / HIST_AVG_PE * 100 if pd.notna(x) else None
    )
    df["NTM P/E vs Hist Avg (%)"] = df["NTM P/E"].apply(
        lambda x: (x - HIST_AVG_PE) / HIST_AVG_PE * 100 if pd.notna(x) else None
    )

    def _flag(ntm):
        if pd.isna(ntm):            return "N/A"
        if ntm > PE_HIGH_BOUND:     return "Above Avg"
        if ntm < PE_LOW_BOUND:      return "Below Avg"
        return "Within Avg"

    df["Valuation Flag"] = df["NTM P/E"].apply(_flag)

    # 6. Drop helper cols
    df.drop(columns=[c for c in ("row_no","Country") if c in df.columns],
            inplace=True, errors="ignore")

    # 7. Deduplicate by ticker
    df.drop_duplicates(subset=["Ticker"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


## Step 4: Sector Aggregations
Groups the companies by GICS Sector and Sub-Industry, computing the Market-Cap Weighted Averages for P/E to visualize broader market trends.


In [4]:
def _wavg(grp: pd.DataFrame, val_col: str, wt: str = "Market Cap ($M)") -> float | None:
    v = grp.dropna(subset=[val_col, wt])
    v = v[v[val_col] > 0]
    if v.empty:
        return None
    return (v[val_col] * v[wt]).sum() / v[wt].sum()

def build_summary(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    total_mc = df["Market Cap ($M)"].sum(skipna=True)
    rows = []
    for name, grp in df.groupby(group_col, sort=True):
        mc   = grp["Market Cap ($M)"].sum(skipna=True)
        vn   = grp["NTM P/E"].dropna(); vn = vn[vn > 0]
        vl   = grp["LTM P/E"].dropna(); vl = vl[vl > 0]
        wntm = _wavg(grp, "NTM P/E")
        rows.append({
            group_col:                    name,
            "No. of Companies":           len(grp),
            "% of Index":                 mc / total_mc * 100 if total_mc else None,
            "Weighted Avg LTM P/E":       _wavg(grp, "LTM P/E"),
            "Weighted Avg NTM P/E":       wntm,
            "Weighted Avg EPS TTM": _wavg(grp, "EPS TTM"),
            "Weighted Avg CQ+1 EPS": _wavg(grp, "CQ+1 EPS Estimate"),

            "Weighted Avg Div Yield (%)": _wavg(grp, "Dividend Yield (%)"),
            "Total Market Cap ($M)":      mc,
            "Median LTM P/E":             float(vl.median()) if len(vl) else None,
            "Median NTM P/E":             float(vn.median()) if len(vn) else None,
            "Min NTM P/E":                float(vn.min())    if len(vn) else None,
            "Max NTM P/E":                float(vn.max())    if len(vn) else None,
            "Valuation Flag": (
                "N/A"        if pd.isna(wntm) else
                "Above Avg"  if wntm > PE_HIGH_BOUND else
                "Below Avg"  if wntm < PE_LOW_BOUND else
                "Within Avg"
            ),
        })
    return (pd.DataFrame(rows)
              .sort_values("Weighted Avg NTM P/E", ascending=False, na_position="last")
              .reset_index(drop=True))


## Step 5: Excel Workbook Builder
Transforms the Pandas DataFrames into a formatted, multi-sheet `.xlsx` file using `openpyxl`. Applies thematic colors, conditional formatting for Valuation Flags, and generates a Bar Chart.


In [5]:

# Palette
NAVY     = "1B2A4A"
WHITE    = "FFFFFF"
ROW_EVEN = "F0F3F7"
ROW_ODD  = "FFFFFF"
R_FILL = "FADBD8"; R_FONT = "922B21"
Y_FILL = "FEF9E7"; Y_FONT = "7D6608"
G_FILL = "D5F5E3"; G_FONT = "1E8449"
N_FILL = "EAECEE"; N_FONT = "717D7E"

def make_labels(as_of: datetime) -> tuple[str, str]:
    """Return (title_text, footer_text) stamped with the given as-of date."""
    d = as_of.strftime("%B %d, %Y")
    title = (
        f"S&P 500 Valuation Analysis — As of {d}  |  "
        "ECM Research  |  Bulge Bracket Bank"
    )
    footer = (
        "Source: Finviz.com Screener | Multpl.com Historical Data | "
        f"Data as of {d}.  "
        "For internal use only. Negative / unavailable P/E multiples excluded.  "
        f"Historical benchmark: {HIST_AVG_PE}x (Scraped S&P 500 Long-Term Median TTM P/E). "
        f"Valuation Bands: Below Avg (<{PE_LOW_BOUND}x) | Within Avg | Above Avg (>{PE_HIGH_BOUND}x)."
    )
    return title, footer


_thin = lambda c="CCCCCC": Side(style="thin", color=c)
DATA_BORDER = Border(left=_thin(), right=_thin(), top=_thin(), bottom=_thin())

def _fill(h): return PatternFill("solid", start_color=h)
def _font(bold=False, size=9, color="000000", italic=False):
    return Font(name="Arial", bold=bold, size=size, color=color, italic=italic)
def _align(h="center", v="center", wrap=False, indent=0):
    return Alignment(horizontal=h, vertical=v, wrap_text=wrap, indent=indent)
def _row_fill(i): return _fill(ROW_EVEN if i % 2 == 0 else ROW_ODD)

def _flag_styles(flag):
    return {
        "Above Avg":  (_fill(R_FILL), _font(bold=True, color=R_FONT)),
        "Within Avg": (_fill(Y_FILL), _font(bold=True, color=Y_FONT)),
        "Below Avg":  (_fill(G_FILL), _font(bold=True, color=G_FONT)),
    }.get(flag, (_fill(N_FILL), _font(bold=True, color=N_FONT)))

def _ntm_fill(v):
    if pd.isna(v): return _fill(N_FILL)
    return _fill(R_FILL if v > PE_HIGH_BOUND else G_FILL if v < PE_LOW_BOUND else Y_FILL)

def _write_title(ws, txt, ncols, row=1):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=ncols)
    c = ws.cell(row=row, column=1, value=txt)
    c.font = _font(bold=True, size=12, color=WHITE)
    c.fill = _fill(NAVY)
    c.alignment = _align(h="left", indent=1)
    ws.row_dimensions[row].height = 24

def _write_headers(ws, hdrs, row=2):
    ws.row_dimensions[row].height = 36
    hb = Border(left=_thin("4A5568"), right=_thin("4A5568"),
                top=_thin("4A5568"), bottom=_thin("4A5568"))
    for ci, h in enumerate(hdrs, 1):
        c = ws.cell(row=row, column=ci, value=h)
        c.font = _font(bold=True, color=WHITE, size=9)
        c.fill = _fill(NAVY)
        c.alignment = _align(wrap=True)
        c.border = hb

def _write_footer(ws, txt, ncols, last_row):
    fr = last_row + 2
    ws.merge_cells(start_row=fr, start_column=1, end_row=fr, end_column=ncols)
    c = ws.cell(row=fr, column=1, value=txt)
    c.font = _font(italic=True, size=7, color="888888")
    c.alignment = _align(h="left")

def _col_widths(ws, widths):
    for ci, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(ci)].width = w

def _apply_fmt(cell, col, val):
    """Apply number formats for Excel."""
    if col in ("Market Cap ($M)", "Total Market Cap ($M)"):
        cell.number_format = '$#,##0'
    elif col in ("Share Price",):
        cell.number_format = '$#,##0.00'
    elif col in ("% of S&P 500 Index","% of Index",
                 "Dividend Yield (%)","Weighted Avg Div Yield (%)"):
        cell.number_format = '0.00%'
        if pd.notna(val): cell.value = val / 100
    elif any(x in col for x in ("P/E","Avg LTM P/E","Avg NTM P/E",
                                "Median LTM","Median NTM","Min NTM","Max NTM")):
        cell.number_format = '0.0"x"'
    elif "Hist Avg" in col:
        cell.number_format = '+0.0%;-0.0%;"-"'
        if pd.notna(val): cell.value = val / 100
    elif "EPS" in col:
        cell.number_format = '0.00'
    elif col == "No. of Companies":
        cell.number_format = '0'


In [6]:
S1_HDRS = [
    "Ticker","Company Name","GICS Sector","GICS Sub-Industry",
    "Market Cap ($M)","% of S&P 500 Index",
    # "Share Price","Change (%)","Volume",
    "LTM P/E","NTM P/E",
    "Dividend Yield (%)",
    "EPS TTM","CQ+1 EPS Estimate",
    "LTM P/E vs Hist Avg (%)","NTM P/E vs Hist Avg (%)",
    "Valuation Flag",
]

S1_COLS = [
    "Ticker","Company Name","GICS Sector","GICS Sub-Industry",
    "Market Cap ($M)","% of S&P 500 Index",
    # "Share Price","Change (%)","Volume",
    "LTM P/E","NTM P/E",
    "Dividend Yield (%)",
    "EPS TTM","CQ+1 EPS Estimate",
    "LTM P/E vs Hist Avg (%)","NTM P/E vs Hist Avg (%)",
    "Valuation Flag",
]


def build_master(wb, df, title_txt, footer_txt):
    ws = wb.active
    ws.title = "S&P500 Master Table"
    _write_title(ws, title_txt, len(S1_HDRS))
    _write_headers(ws, S1_HDRS)
    for di, (_, row) in enumerate(df.iterrows(), 1):
        ri   = di + 2
        flag = row.get("Valuation Flag","N/A")
        bf   = _row_fill(di)
        for ci, col in enumerate(S1_COLS, 1):
            val = row.get(col)
            c   = ws.cell(row=ri, column=ci, value=val)
            c.font   = _font(size=9)
            c.border = DATA_BORDER
            c.alignment = _align(h="left" if ci <= 2 else "center")
            if   col == "NTM P/E":       c.fill = _ntm_fill(row.get("NTM P/E"))
            elif col == "Valuation Flag": c.fill, c.font = _flag_styles(flag)
            else:                         c.fill = bf
            _apply_fmt(c, col, val)
    lr = 2 + len(df)
    ws.freeze_panes = "A3"
    ws.auto_filter.ref = f"A2:{get_column_letter(len(S1_HDRS))}{lr}"
    _write_footer(ws, footer_txt, len(S1_HDRS), lr)
    _col_widths(ws, [8, 32, 24, 36, 14, 11, 8, 8, 12, 10, 10, 18, 18, 13])



In [7]:
S2_HDRS = ["Name","% of Index","# Companies",
           "Wtd Avg LTM P/E","Wtd Avg NTM P/E",
           "Wtd Avg Div Yield (%)","Total Mkt Cap ($M)",
           "Median LTM P/E","Median NTM P/E",
           "Min NTM P/E","Max NTM P/E","Valuation Flag"]

S2_COLS_TEMPLATE = [None,
                    "% of Index","No. of Companies",
                    "Weighted Avg LTM P/E","Weighted Avg NTM P/E","Weighted Avg Div Yield (%)",
                    "Total Market Cap ($M)","Median LTM P/E","Median NTM P/E",
                    "Min NTM P/E","Max NTM P/E","Valuation Flag"]

def build_summary_sheet(wb, sdf, group_col, sheet_name, title_txt, footer_txt, add_chart=False):
    ws = wb.create_sheet(sheet_name)
    _write_title(ws, title_txt, len(S2_HDRS))
    _write_headers(ws, S2_HDRS)
    cols = [group_col] + S2_COLS_TEMPLATE[1:]
    for di, (_, row) in enumerate(sdf.iterrows(), 1):
        ri   = di + 2
        flag = row.get("Valuation Flag","N/A")
        bf   = _row_fill(di)
        for ci, col in enumerate(cols, 1):
            val = row.get(col)
            c   = ws.cell(row=ri, column=ci, value=val)
            c.font   = _font(size=9)
            c.border = DATA_BORDER
            c.alignment = _align(h="left" if ci == 1 else "center")
            if   col == "Weighted Avg NTM P/E": c.fill = _ntm_fill(row.get("Weighted Avg NTM P/E"))
            elif col == "Valuation Flag":        c.fill, c.font = _flag_styles(flag)
            else:                                c.fill = bf
            _apply_fmt(c, col, val)
    lr = 2 + len(sdf)
    ws.freeze_panes = "A3"
    ws.auto_filter.ref = f"A2:{get_column_letter(len(S2_HDRS))}{lr}"
    _write_footer(ws, footer_txt, len(S2_HDRS), lr)
    _col_widths(ws, [36,10,12,14,14,16,18,13,13,11,11,13])
    if add_chart:
        _add_chart(ws, sdf, group_col, lr)

def _add_chart(ws, sdf, group_col, lr):
    cr = lr + 4
    ws.cell(row=cr-1, column=1,
            value="Weighted Avg NTM P/E by Sector vs. Historical Average").font = \
        _font(bold=True, size=11, color=NAVY)
    ws.cell(row=cr, column=1, value="Sector").font = _font(bold=True)
    ws.cell(row=cr, column=2, value="NTM P/E").font = _font(bold=True)
    ws.cell(row=cr, column=3, value=f"Hist Avg ({HIST_AVG_PE}x)").font = _font(bold=True)
    for i, (_, row) in enumerate(sdf.iterrows(), 1):
        rr  = cr + i
        ntm = row.get("Weighted Avg NTM P/E")
        ws.cell(row=rr, column=1, value=row.get(group_col, row.iloc[0]))
        ws.cell(row=rr, column=2, value=round(ntm, 2) if pd.notna(ntm) else 0)
        ws.cell(row=rr, column=3, value=HIST_AVG_PE)
    n     = len(sdf)
    chart = BarChart()
    chart.type="col"; chart.grouping="clustered"
    chart.title        = "Weighted Avg NTM P/E by GICS Sector vs. Historical Average"
    chart.y_axis.title = "NTM P/E (x)"
    chart.x_axis.title = "GICS Sector"
    chart.style=10; chart.width=30; chart.height=16
    chart.add_data(Reference(ws, min_col=2, max_col=3,
                             min_row=cr, max_row=cr+n), titles_from_data=True)
    chart.set_categories(Reference(ws, min_col=1, min_row=cr+1, max_row=cr+n))
    chart.series[0].graphicalProperties.solidFill = "2E4057"
    chart.series[1].graphicalProperties.solidFill = "E74C3C"
    ws.add_chart(chart, f"A{cr+2}")


In [8]:
def _section_header(ws, title, color, ncols, row):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=ncols)
    c = ws.cell(row=row, column=1, value=title)
    c.font = _font(bold=True, color=WHITE, size=10)
    c.fill = _fill(color)
    c.alignment = _align()
    ws.row_dimensions[row].height = 22

def _subtable(ws, title, color, hdrs, rows, start_row):
    _section_header(ws, title, color, len(hdrs), start_row)
    hr = start_row + 1
    ws.row_dimensions[hr].height = 28
    hb = Border(left=_thin("4A5568"), right=_thin("4A5568"),
                top=_thin("4A5568"),  bottom=_thin("4A5568"))
    for ci, h in enumerate(hdrs, 1):
        c = ws.cell(row=hr, column=ci, value=h)
        c.font = _font(bold=True, color=WHITE, size=9)
        c.fill = _fill(NAVY); c.alignment = _align(wrap=True); c.border = hb
    for ri_off, rd in enumerate(rows):
        ri = hr + 1 + ri_off
        for ci, val in enumerate(rd, 1):
            c = ws.cell(row=ri, column=ci, value=val)
            c.font = _font(size=9); c.border = DATA_BORDER
            c.alignment = _align(h="left" if ci == 1 else "center")
            c.fill = _row_fill(ri_off + 1)
    return hr + 1 + len(rows)

def _co_rows(subset):
    out = []
    for _, r in subset.iterrows():
        out.append([
            r.get("Ticker",""),
            r.get("Company Name",""),
            r.get("GICS Sector",""),
            f"{r['NTM P/E']:.1f}x",
            f"{r['LTM P/E']:.1f}x" if pd.notna(r.get("LTM P/E")) else "N/A",
            f"{r['Dividend Yield (%)']:.2f}%" if pd.notna(r.get("Dividend Yield (%)")) else "N/A",
            f"${r['Market Cap ($M)']:,.0f}M" if pd.notna(r.get("Market Cap ($M)")) else "N/A",
        ])
    return out

def _sec_rows(subset):
    out = []
    for _, r in subset.iterrows():
        wntm = r.get("Weighted Avg NTM P/E")
        wltm = r.get("Weighted Avg LTM P/E")
        mc   = r.get("Total Market Cap ($M)")
        out.append([
            r.iloc[0],
            f"{wntm:.1f}x" if pd.notna(wntm) else "N/A",
            f"{wltm:.1f}x" if pd.notna(wltm) else "N/A",
            int(r.get("No. of Companies",0)),
            f"${mc:,.0f}M"  if pd.notna(mc)  else "N/A",
        ])
    return out

CO_H  = ["Ticker","Company","GICS Sector","NTM P/E","LTM P/E","Div Yield","Mkt Cap ($M)"]
SEC_H = ["Sub-Industry / Sector","Wtd Avg NTM P/E","Wtd Avg LTM P/E","# Co.","Total Mkt Cap"]

def build_extremes(wb, df, sub_df, title_txt, footer_txt):
    ws  = wb.create_sheet("Extremes Dashboard")
    _write_title(ws, title_txt, 7)
    valid = df[df["NTM P/E"].notna() & (df["NTM P/E"] > 0)]
    vsub  = sub_df[sub_df["Weighted Avg NTM P/E"].notna()]
    cur = 3
    cur = _subtable(
        ws,
        "🔴  TOP 25 MOST EXPENSIVE COMPANIES  —  By NTM P/E  (Potential Over-Enthusiasm)",
        "C0392B", CO_H, _co_rows(valid.nlargest(25,"NTM P/E")), cur
    ) + 2
    cur = _subtable(
        ws,
        "🟢  TOP 25 CHEAPEST COMPANIES  —  By NTM P/E, Excl. Negatives  (Potential Over-Selling)",
        "1E8449", CO_H, _co_rows(valid.nsmallest(25,"NTM P/E")), cur
    ) + 2
    cur = _subtable(
        ws,
        "🔴  TOP 25 MOST EXPENSIVE SUB-SECTORS  —  Wtd Avg NTM P/E",
        "7B241C", SEC_H, _sec_rows(vsub.head(25)), cur
    ) + 2
    cur = _subtable(
        ws,
        "🟢  TOP 25 CHEAPEST SUB-SECTORS  —  Wtd Avg NTM P/E",
        "1A5276", SEC_H, _sec_rows(vsub.nsmallest(25,"Weighted Avg NTM P/E")), cur
    ) + 2
    _write_footer(ws, footer_txt, 7, cur)
    _col_widths(ws, [8,32,26,10,10,12,14])

def build_workbook(df, sub_df, sec_df, as_of: datetime) -> Workbook:
    title_txt, footer_txt = make_labels(as_of)
    wb = Workbook()
    log.info("Sheet 1: S&P500 Master Table")
    build_master(wb, df, title_txt, footer_txt)
    log.info("Sheet 2: Sub-Sector Summary")
    build_summary_sheet(wb, sub_df, "GICS Sub-Industry", "Sub-Sector Summary",
                        title_txt, footer_txt)
    log.info("Sheet 3: Sector Summary + chart")
    build_summary_sheet(wb, sec_df, "GICS Sector", "Sector Summary",
                        title_txt, footer_txt, add_chart=True)
    log.info("Sheet 4: Extremes Dashboard")
    build_extremes(wb, df, sub_df, title_txt, footer_txt)
    return wb


## Step 6: Console Overview & Report Generation
Runs the entire pipeline. 

Scraping Finviz -> Standardizing Data -> Grouping -> Printing Summary -> Saving Excel.


## ✨ Step 7: Claude API Setup

API key is loaded from `.env` — never hardcoded in the notebook.
Client setup matches `claude_api.ipynb` exactly:
- `load_dotenv()` → `Anthropic()` (no key argument)
- `model = 'claude-sonnet-4-0'`
- `chat(messages, system, temperature)` function
- `add_user_message()` / `add_assistant_message()` helpers

**One-time setup:** Create `.env` next to this notebook with:
```
ANTHROPIC_API_KEY=sk-ant-your-key-here
```


In [9]:
%pip install -q anthropic python-dotenv networkx

from dotenv import load_dotenv
load_dotenv()  # reads ANTHROPIC_API_KEY from .env — key never hardcoded

from anthropic import Anthropic
import os, json as _json, re, networkx as nx

# ── Matches claude_api.ipynb exactly ─────────────────────────────────────────
client = Anthropic()           # auto-reads ANTHROPIC_API_KEY from environment
model  = "claude-sonnet-4-0"  # same model as claude_api.ipynb

def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=0.3, max_tokens=1000):
    """
    Core chat function — matches claude_api.ipynb signature exactly.
    messages    : list built with add_user_message / add_assistant_message
    system      : optional system prompt string
    temperature : 0.3 default keeps financial outputs focused
    """
    params = {
        "model":       model,
        "max_tokens":  max_tokens,
        "messages":    messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    message = client.messages.create(**params)
    return message.content[0].text

# Smoke test
try:
    _msgs = []
    add_user_message(_msgs, "Reply with exactly two words: API ready")
    _resp = chat(_msgs, max_tokens=10)
    print(f"✅ Claude connected — model : {model}")
    print(f"   Test response     : '{_resp.strip()}'")
    print(f"   Key source        : .env file")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   → Check .env contains: ANTHROPIC_API_KEY=sk-ant-...")



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\shreyansh\AppData\Local\Temp\ipykernel_6084\2414856187.py:34: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  message = client.messages.create(**params)
20:27:00  INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


✅ Claude connected — model : claude-sonnet-4-0
   Test response     : 'API ready'
   Key source        : .env file


### Gen AI Feature 1 — AI Executive Commentary

Claude reads live sector P/E aggregates and writes a 150-word analyst note
injected into the Excel report as a **'Market Intelligence'** block.
Uses a single-turn `chat()` call with a financial analyst system prompt.


In [10]:
def generate_ai_commentary(sec_df, hist_pe: float) -> str:
    """
    FEATURE 1 — AI Executive Commentary
    Claude receives sector P/E data and returns a professional analyst note.
    """
    pe_col = next((c for c in ["Wtd Avg NTM P/E", "Wtd Avg LTM P/E", "NTM P/E", "LTM P/E"]
                   if c in sec_df.columns), None)
    if pe_col is None or sec_df.empty:
        return "[Insufficient sector data for AI commentary]"

    name_col   = "Name" if "Name" in sec_df.columns else sec_df.columns[0]
    top3_exp   = sec_df.dropna(subset=[pe_col]).nlargest(3, pe_col)[[name_col, pe_col]].to_string(index=False)
    top3_cheap = sec_df.dropna(subset=[pe_col]).nsmallest(3, pe_col)[[name_col, pe_col]].to_string(index=False)
    bench      = f"{hist_pe:.1f}x"

    prompt = (
        "Write a concise 150-word executive commentary for an internal S&P 500 valuation dashboard.\n\n"
        f"Historical Median P/E Benchmark: {bench}\n\n"
        f"Top 3 Most Expensive Sectors (NTM P/E):\n{top3_exp}\n\n"
        f"Top 3 Cheapest Sectors (NTM P/E):\n{top3_cheap}\n\n"
        f"Rules:\n"
        f"- Identify sectors at premium or discount to the {bench} median\n"
        "- Note 1-2 key investment themes\n"
        "- Professional equity research tone\n"
        "- Output only the commentary paragraph, no heading"
    )

    msgs = []
    add_user_message(msgs, prompt)
    result = chat(
        msgs,
        system="You are a senior equity research analyst at a bulge bracket bank.",
        temperature=0.3
    )
    print("✅ AI Commentary Generated:")
    print("─" * 60)
    print(result)
    print("─" * 60)
    return result

try:
    AI_COMMENTARY = generate_ai_commentary(sec_df, HIST_AVG_PE)
except NameError:
    AI_COMMENTARY = "[Run Step 4 first to generate sec_df]"
    print("⚠️  Run Step 4 (Sector Aggregations) first, then re-run this cell.")


⚠️  Run Step 4 (Sector Aggregations) first, then re-run this cell.


### Gen AI Feature 2 — Natural Language Query (NLQ) Interface

A banker types a plain-English question. Claude translates it to a Pandas query,
executes it, and explains the result. Uses a two-turn `chat()` conversation:
Turn 1 generates the query string; Turn 2 summarises the results.


In [11]:
def natural_language_query(question: str, dataframe=None) -> str:
    """
    FEATURE 2 — Natural Language Query Interface
    Two-turn Claude conversation:
      Turn 1: question -> pandas query string
      Turn 2: query results -> natural language summary
    """
    if dataframe is None:
        try:
            dataframe = df
        except NameError:
            return "⚠️  No DataFrame available — run Steps 2-3 first."

    col_info = ", ".join([f"'{c}' ({str(dataframe[c].dtype)})" for c in dataframe.columns[:15]])

    # Turn 1 — generate Pandas query
    msgs = []
    add_user_message(msgs,
        f"You are a Pandas expert. Translate this into a df.query() string.\n\n"
        f"DataFrame columns: {col_info}\n"
        f"Question: \"{question}\"\n\n"
        "Output ONLY the query string. No backticks, no explanation."
    )
    raw_query = chat(
        msgs,
        system="You are a Pandas expert. Output only valid df.query() strings.",
        temperature=0.1,
        max_tokens=80
    ).strip().strip("`'\"")

    safe_query = re.sub(r"[^a-zA-Z0-9_\s\.\>\<\=\!\(\)\'\"\&\|\-%@\$\#\[\]]", "", raw_query)

    try:
        result_df = dataframe.query(safe_query)
        n         = len(result_df)
        preview   = result_df.head(5).to_string(index=False) if n > 0 else "No results found."

        # Turn 2 — summarise results (maintain conversation history)
        add_assistant_message(msgs, raw_query)
        add_user_message(msgs,
            f"The query returned {n} companies. Top results:\n{preview}\n\n"
            "Write a 2-sentence answer. Be specific — cite company names, sectors, P/E values."
        )
        summary = chat(msgs, temperature=0.3)
        print(f"\n🤖 Q: {question}")
        print(f"   Query   : {safe_query}")
        print(f"   Matched : {n} companies")
        print(f"   Answer  : {summary}\n")
        return summary

    except Exception as e:
        fallback_msgs = []
        add_user_message(fallback_msgs, f"Answer this S&P 500 question: {question}")
        fallback = chat(fallback_msgs, temperature=0.3)
        print(f"⚠️  Query failed ({e}). Direct answer: {fallback}")
        return fallback


# Demo
try:
    print("=" * 65)
    print("NATURAL LANGUAGE QUERY DEMO")
    print("=" * 65)
    for q in [
        "Show me the top 10 most undervalued companies by NTM P/E",
        "Which Technology companies have a dividend yield above 1%?",
        "What are the 5 largest companies by market cap?",
    ]:
        natural_language_query(q, df)
except NameError:
    print("⚠️  df not found — run Steps 2-3 first.")


NATURAL LANGUAGE QUERY DEMO
⚠️  df not found — run Steps 2-3 first.


### Gen AI Feature 3 — AI Qualitative Risk Flags

For the 10 most extreme P/E outliers, Claude writes a one-sentence
sector-contextualised risk explanation using the `chat()` pattern.


In [12]:
def generate_risk_flags(dataframe, top_n: int = 10) -> dict:
    """
    FEATURE 3 — AI Qualitative Risk Flags
    Single-turn chat() call per outlier company.
    """
    pe_col = "NTM P/E" if "NTM P/E" in dataframe.columns else "LTM P/E"
    valid  = dataframe.dropna(subset=[pe_col]).copy()
    if valid.empty:
        print("⚠️  No valid P/E data found.")
        return {}

    outliers   = pd.concat([valid.nlargest(top_n // 2, pe_col), valid.nsmallest(top_n // 2, pe_col)])
    risk_flags = {}
    print(f"\n🤖 Generating AI risk flags for {len(outliers)} extreme outliers...\n")

    for _, row in outliers.iterrows():
        ticker  = row.get("Ticker", "N/A")
        company = row.get("Company Name", ticker)
        sector  = row.get("GICS Sector", "Unknown")
        pe_val  = row.get(pe_col, 0)
        flag    = row.get("Valuation Flag", "N/A")

        msgs = []
        add_user_message(msgs,
            f"In one sentence (max 20 words), state the key valuation risk for:\n"
            f"Company: {company} ({ticker}) | Sector: {sector} | "
            f"NTM P/E: {pe_val:.1f}x | Flag: {flag}\n\n"
            "Be direct and sector-specific. No intro phrases."
        )
        explanation = chat(
            msgs,
            system="You are a concise equity research analyst. One sentence only.",
            temperature=0.3,
            max_tokens=60
        )
        risk_flags[ticker] = explanation
        print(f"  [{flag:^10}] {ticker:6} ({pe_val:5.1f}x): {explanation}")

    return risk_flags


try:
    AI_RISK_FLAGS = generate_risk_flags(df, top_n=10)
    if AI_RISK_FLAGS:
        df["AI Risk Note"] = df["Ticker"].map(AI_RISK_FLAGS).fillna("")
        print(f"\n✅ 'AI Risk Note' column added ({len(AI_RISK_FLAGS)} companies flagged).")
except NameError:
    print("⚠️  df not found — run Steps 2-3 first.")


⚠️  df not found — run Steps 2-3 first.


### Gen AI Feature 4 — Agentic RAG

Claude drives a multi-turn tool-use loop using the `add_user_message` /
`add_assistant_message` / `chat()` pattern from `claude_api.ipynb`.
It picks tools, reads results, and decides when to stop.

| Tool | What it does |
|------|--------------|
| `query_sp500_data` | Filter companies by sector/flag |
| `get_sector_summary` | Weighted P/E aggregates by sector |
| `flag_outliers` | Most over/undervalued companies |
| `compare_sectors` | Two-sector head-to-head |


In [13]:
# ── Tool functions ───────────────────────────────────────────────────────────
def _tool_query_sp500(sector=None, flag=None, top_n=10):
    try:
        result = df.copy()
        if sector: result = result[result["GICS Sector"].str.contains(sector, case=False, na=False)]
        if flag:   result = result[result["Valuation Flag"] == flag]
        cols = [c for c in ["Ticker","Company Name","GICS Sector","NTM P/E","Valuation Flag","Market Cap ($M)"]
                if c in result.columns]
        return result.nlargest(int(top_n), "Market Cap ($M)")[cols].to_string(index=False)
    except Exception as e:
        return f"Error: {e}"

def _tool_sector_summary(sector=None):
    try:
        s        = sec_df.copy()
        name_col = "Name" if "Name" in s.columns else s.columns[0]
        if sector: s = s[s[name_col].str.contains(sector, case=False, na=False)]
        cols = [c for c in ["Name","Wtd Avg NTM P/E","Wtd Avg LTM P/E","# Companies","% of Index"] if c in s.columns]
        return s[cols].to_string(index=False)
    except Exception as e:
        return f"Error: {e}"

def _tool_flag_outliers(direction="expensive", top_n=10):
    try:
        col    = "NTM P/E" if "NTM P/E" in df.columns else "LTM P/E"
        v      = df.dropna(subset=[col])
        result = v.nlargest(int(top_n), col) if direction == "expensive" else v.nsmallest(int(top_n), col)
        cols   = [c for c in ["Ticker","Company Name","GICS Sector","NTM P/E","Valuation Flag"] if c in result.columns]
        return result[cols].to_string(index=False)
    except Exception as e:
        return f"Error: {e}"

def _tool_compare_sectors(sector_a, sector_b):
    try:
        name_col = "Name" if "Name" in sec_df.columns else sec_df.columns[0]
        rows     = [sec_df[sec_df[name_col].str.contains(s, case=False, na=False)] for s in [sector_a, sector_b]]
        return pd.concat(rows).to_string(index=False)
    except Exception as e:
        return f"Error: {e}"

AGENT_TOOLS = {
    "query_sp500_data":   _tool_query_sp500,
    "get_sector_summary": _tool_sector_summary,
    "flag_outliers":      _tool_flag_outliers,
    "compare_sectors":    _tool_compare_sectors,
}

TOOL_SCHEMA = (
    "Available tools — respond with JSON only:\n"
    "- query_sp500_data(sector, flag, top_n)\n"
    "- get_sector_summary(sector)\n"
    "- flag_outliers(direction='expensive'|'cheap', top_n)\n"
    "- compare_sectors(sector_a, sector_b)\n\n"
    'To call a tool:    {"tool": "tool_name", "args": {"arg": "val"}}\n'
    'To give an answer: {"answer": "your full natural language answer"}'
)

def run_agent(question: str, max_turns: int = 5) -> str:
    """
    Agentic RAG — Claude drives a multi-turn tool loop.
    Uses add_user_message / add_assistant_message / chat() pattern.
    """
    print(f"\n{'='*65}")
    print(f"🤖 AGENT: {question}")
    print(f"{'='*65}")

    messages = []
    add_user_message(messages,
        TOOL_SCHEMA + f'\n\nQuestion: "{question}"\nStart by calling the most relevant tool.'
    )

    for turn in range(max_turns):
        raw       = chat(messages,
                         system="You are a financial analyst agent. Always respond with valid JSON only.",
                         temperature=0.1)
        raw_clean = re.sub(r"```(?:json)?\n?|```", "", raw).strip()
        add_assistant_message(messages, raw)

        try:
            parsed = _json.loads(raw_clean)
            if "answer" in parsed:
                print(f"\n💬 Answer: {parsed['answer']}")
                return parsed["answer"]

            tool_name = parsed.get("tool")
            tool_args = parsed.get("args", {})
            if tool_name not in AGENT_TOOLS:
                return f"Agent called unknown tool: {tool_name}"

            tool_result = AGENT_TOOLS[tool_name](**tool_args)
            print(f"  Turn {turn+1} → {tool_name}({tool_args})")
            add_user_message(messages,
                f"Tool '{tool_name}' returned:\n{tool_result}\n\n"
                f'Original question: "{question}"\n'
                "Continue: call another tool or give your final JSON answer."
            )
        except _json.JSONDecodeError:
            print(f"\n💬 Answer: {raw}")
            return raw

    return "Agent reached max turns."


# Demo
try:
    run_agent("Which sectors are most overvalued vs historical median P/E?")
    run_agent("Find undervalued Healthcare companies with the largest market caps")
except NameError:
    print("⚠️  df / sec_df not found — run Steps 2-4 first.")


C:\Users\shreyansh\AppData\Local\Temp\ipykernel_6084\2414856187.py:34: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  message = client.messages.create(**params)



🤖 AGENT: Which sectors are most overvalued vs historical median P/E?


20:27:01  INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


  Turn 1 → flag_outliers({'direction': 'expensive', 'top_n': 10})


20:27:03  INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


...

  Turn 2 → get_sector_summary({'sector': 'Health Care'})


20:27:13  INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


  Turn 3 → flag_outliers({'direction': 'cheap', 'top_n': 10})


20:27:17  INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



💬 Answer: I apologize, but I'm unable to retrieve the requested information about undervalued Healthcare companies with the largest market caps. All available tools are currently experiencing technical issues with undefined data variables ('df' and 'sec_df' are not defined). This suggests the underlying data source may not be properly loaded or initialized. To get this information, you would need the data infrastructure to be properly set up first, then I could help identify undervalued Healthcare sector companies ranked by market capitalization.


### Gen AI Feature 5 — Graph RAG

Builds an in-memory NetworkX knowledge graph from the S&P 500 DataFrame.
Claude reasons over the structured subgraph using `chat()` from `claude_api.ipynb`.

**Graph schema:**
```
(AAPL) -[BELONGS_TO]→  (Technology)
(AAPL) -[HAS_FLAG]→    (Above Avg)
(NVDA) -[PEER_OF]→     (AMD)
(Technology) -[AVG_PE]→ (28.4)
```


In [14]:
def build_sp500_graph(dataframe) -> nx.DiGraph:
    """
    Converts S&P 500 DataFrame into a directed knowledge graph.
    Nodes : Companies, Sectors, Industries, Valuation Flags
    Edges : belongs_to, in_industry, has_flag, peer_of
    """
    G = nx.DiGraph()
    sector_companies = {}

    for _, row in dataframe.iterrows():
        ticker   = str(row.get("Ticker", "?"))
        sector   = str(row.get("GICS Sector", "Unknown"))
        industry = str(row.get("GICS Sub-Industry", "Unknown"))
        flag     = str(row.get("Valuation Flag", "N/A"))
        pe       = row.get("NTM P/E") or row.get("LTM P/E")
        mktcap   = row.get("Market Cap ($M)")

        G.add_node(ticker,   type="company", name=row.get("Company Name", ticker),
                   ntm_pe=pe, market_cap=mktcap, flag=flag, sector=sector)
        G.add_node(sector,   type="sector")
        G.add_node(industry, type="industry")
        G.add_node(flag,     type="valuation_flag")
        G.add_edge(ticker, sector,   relation="belongs_to")
        G.add_edge(ticker, industry, relation="in_industry")
        G.add_edge(ticker, flag,     relation="has_flag")
        sector_companies.setdefault(sector, []).append(ticker)

    for sector, peers in sector_companies.items():
        for i, a in enumerate(peers):
            for b in peers[i+1:i+4]:
                G.add_edge(a, b, relation="peer_of")
    return G


def graph_rag_query(G: nx.DiGraph, question: str) -> str:
    """
    Graph RAG — extracts structured subgraph context and passes it
    to Claude via the chat() pattern from claude_api.ipynb.
    """
    sector_stats = {}
    for node, data in G.nodes(data=True):
        if data.get("type") == "company":
            pe  = data.get("ntm_pe")
            sec = data.get("sector", "Unknown")
            flg = data.get("flag", "N/A")
            if pe and isinstance(pe, (int, float)):
                sector_stats.setdefault(sec, {"pe_vals": [], "flags": []})
                sector_stats[sec]["pe_vals"].append(pe)
                sector_stats[sec]["flags"].append(flg)

    graph_context = "\n".join([
        f"  {sec}: avg NTM P/E={sum(v['pe_vals'])/len(v['pe_vals']):.1f}x | "
        f"{len(v['pe_vals'])} companies | "
        f"{v['flags'].count('Above Avg')} above avg | "
        f"{v['flags'].count('Below Avg')} below avg"
        for sec, v in sorted(sector_stats.items())
    ])

    top5 = sorted(
        [(n, d) for n, d in G.nodes(data=True) if d.get("type") == "company" and d.get("market_cap")],
        key=lambda x: x[1].get("market_cap", 0), reverse=True
    )[:5]
    top_str = "\n".join([
        f"  {d['name']} ({n}): {d.get('sector')} | NTM P/E={d.get('ntm_pe','N/A')} | Flag={d.get('flag')}"
        for n, d in top5
    ])

    msgs = []
    add_user_message(msgs,
        f"You have access to an S&P 500 knowledge graph.\n\n"
        f"GRAPH — Sector Aggregates:\n{graph_context}\n\n"
        f"GRAPH — Top 5 Companies by Market Cap:\n{top_str}\n\n"
        f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges\n\n"
        f"Answer using ONLY the graph context: \"{question}\"\n"
        "Be specific — cite sector names, P/E values, company names."
    )
    return chat(
        msgs,
        system="You are a financial analyst reasoning over a structured knowledge graph.",
        temperature=0.3
    )


# Build graph and run demos
try:
    print("🔨 Building S&P 500 knowledge graph...")
    SP500_GRAPH = build_sp500_graph(df)
    print(f"✅ Graph: {SP500_GRAPH.number_of_nodes()} nodes, {SP500_GRAPH.number_of_edges()} edges\n")
    for q in [
        "Which sectors have the most companies trading above historical average P/E?",
        "How does Technology compare to Energy in valuation?",
    ]:
        print(f"\n📊 Q: {q}")
        print(f"💬 A: {graph_rag_query(SP500_GRAPH, q)}")
except NameError:
    print("⚠️  df not found — run Steps 2-3 first.")


🔨 Building S&P 500 knowledge graph...
⚠️  df not found — run Steps 2-3 first.


### 🧪 Step 8: Unit Tests

Programmatic correctness tests for all core pipeline functions and the Claude API connection.
Replaces the static rubric with verifiable automated assertions.


In [15]:
def run_tests():
    passed, failed = 0, 0

    def check(name, condition, detail=""):
        nonlocal passed, failed
        if condition:
            print(f"  ✅ PASS: {name}"); passed += 1
        else:
            print(f"  ❌ FAIL: {name}" + (f" — {detail}" if detail else "")); failed += 1

    print("=" * 55)
    print("UNIT TEST SUITE — S&P 500 Valuation Pipeline")
    print("=" * 55)

    print("\n[1] Market Cap Parser")
    check("342.10B → 342,100M",  _parse_market_cap("342.10B") == 342_100.0)
    check("1.5T → 1,500,000M",   _parse_market_cap("1.5T")    == 1_500_000.0)
    check("500M → 500",          _parse_market_cap("500M")    == 500.0)
    check("Dash → None",         _parse_market_cap("-")       is None)
    check("N/A → None",          _parse_market_cap("N/A")     is None)

    print("\n[2] Float Parser")
    check("'25.3' → 25.3",       _parse_float("25.3")    == 25.3)
    check("'1,234.5' → 1234.5",  _parse_float("1,234.5") == 1234.5)
    check("'-' → None",          _parse_float("-")       is None)

    print("\n[3] Percentage Parser")
    check("'2.5%' → 2.5",        _parse_pct("2.5%")  == 2.5)
    check("'0.80%' → 0.8",       _parse_pct("0.80%") == 0.8)
    check("'-' → None",          _parse_pct("-")      is None)

    print("\n[4] Historical P/E Benchmark")
    check("HIST_AVG_PE is numeric",  isinstance(HIST_AVG_PE, (int, float)))
    check("HIST_AVG_PE in 10-25x",   10.0 < HIST_AVG_PE < 25.0, f"Got {HIST_AVG_PE}")
    check("PE_HIGH = PE + 2.5",      abs(PE_HIGH_BOUND - (HIST_AVG_PE + 2.5)) < 0.01)
    check("PE_LOW  = PE - 2.5",      abs(PE_LOW_BOUND  - (HIST_AVG_PE - 2.5)) < 0.01)

    print("\n[5] Valuation Flag Logic")
    def _flag(ntm):
        if pd.isna(ntm):         return "N/A"
        if ntm > PE_HIGH_BOUND:  return "Above Avg"
        if ntm < PE_LOW_BOUND:   return "Below Avg"
        return "Within Avg"
    check("100x → Above Avg",    _flag(100.0) == "Above Avg")
    check("1x → Below Avg",      _flag(1.0)   == "Below Avg")
    check("Median → Within Avg", _flag(HIST_AVG_PE) == "Within Avg")
    check("NaN → N/A",           _flag(float("nan")) == "N/A")

    print("\n[6] DataFrame Integrity")
    try:
        check("df not empty",           len(df) > 0)
        check("Ticker column exists",   "Ticker" in df.columns)
        check("Market Cap exists",      "Market Cap ($M)" in df.columns)
        check("Valuation Flag exists",  "Valuation Flag" in df.columns)
        check("No duplicate tickers",   df["Ticker"].duplicated().sum() == 0,
              f"{df['Ticker'].duplicated().sum()} duplicates")
        check("Market caps positive",   (df["Market Cap ($M)"].dropna() > 0).all())
    except NameError:
        print("  ⚠️  Skipped — run Steps 2-3 first")

    print("\n[7] Claude API (via .env)")
    try:
        _t = []
        add_user_message(_t, "Reply with one word: working")
        resp = chat(_t, max_tokens=10)
        check("Claude API responds",  len(resp) > 0)
        check("Correct model",        model == "claude-sonnet-4-0")
    except Exception as e:
        check("Claude API responds", False, str(e))

    print(f"\n{'='*55}")
    print(f"Results: {passed} passed, {failed} failed / {passed+failed} total")
    print("🎉 All tests passed!" if failed == 0 else f"⚠️  {failed} test(s) need attention.")
    print("=" * 55)

run_tests()


C:\Users\shreyansh\AppData\Local\Temp\ipykernel_6084\2414856187.py:34: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  message = client.messages.create(**params)


UNIT TEST SUITE — S&P 500 Valuation Pipeline

[1] Market Cap Parser
  ✅ PASS: 342.10B → 342,100M
  ✅ PASS: 1.5T → 1,500,000M
  ✅ PASS: 500M → 500
  ✅ PASS: Dash → None
  ✅ PASS: N/A → None

[2] Float Parser
  ✅ PASS: '25.3' → 25.3
  ✅ PASS: '1,234.5' → 1234.5
  ✅ PASS: '-' → None

[3] Percentage Parser
  ✅ PASS: '2.5%' → 2.5
  ✅ PASS: '0.80%' → 0.8
  ✅ PASS: '-' → None

[4] Historical P/E Benchmark
  ✅ PASS: HIST_AVG_PE is numeric
  ✅ PASS: HIST_AVG_PE in 10-25x
  ✅ PASS: PE_HIGH = PE + 2.5
  ✅ PASS: PE_LOW  = PE - 2.5

[5] Valuation Flag Logic
  ✅ PASS: 100x → Above Avg
  ✅ PASS: 1x → Below Avg
  ✅ PASS: Median → Within Avg
  ✅ PASS: NaN → N/A

[6] DataFrame Integrity
  ⚠️  Skipped — run Steps 2-3 first

[7] Claude API (via .env)


20:27:18  INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


  ✅ PASS: Claude API responds
  ✅ PASS: Correct model

Results: 21 passed, 0 failed / 21 total
🎉 All tests passed!


In [16]:
# ─────────────────────────────────────────────────────────────
# STEP 5 — CONSOLE SUMMARY
# ─────────────────────────────────────────────────────────────

def print_summary(df, sub_df, as_of: datetime):
    total    = len(df)
    miss_ntm = int(df["NTM P/E"].isna().sum())
    miss_ltm = int(df["LTM P/E"].isna().sum())
    flags    = df["Valuation Flag"].value_counts()
    total_mc = df["Market Cap ($M)"].sum(skipna=True)

    def wt_pe(pe_col):
        v = df[df[pe_col].notna() & (df[pe_col]>0) & df["Market Cap ($M)"].notna()]
        if v.empty: return float("nan")
        return (v[pe_col]*v["Market Cap ($M)"]).sum() / v["Market Cap ($M)"].sum()

    sp_ntm = wt_pe("NTM P/E"); sp_ltm = wt_pe("LTM P/E")
    vsub   = sub_df[sub_df["Weighted Avg NTM P/E"].notna()]

    print("\n" + "═"*72)
    print("   S&P 500 VALUATION ANALYSIS — CONSOLE SUMMARY")
    print(f"   {as_of:%B %d, %Y}  |  ECM Research")
    print("═"*72)
    print(f"\n  DATA COVERAGE (Finviz S&P 500 screener):")
    print(f"    Companies captured              : {total}")
    print(f"    Missing NTM P/E (no consensus)  : {miss_ntm}  ({miss_ntm/max(total,1)*100:.1f}%)")
    print(f"    Missing LTM P/E (neg. earnings) : {miss_ltm}  ({miss_ltm/max(total,1)*100:.1f}%)")
    print(f"\n  VALUATION FLAGS  (thresholds: <{PE_LOW_BOUND}x = Below, "
          f"{PE_LOW_BOUND}–{PE_HIGH_BOUND}x = Within, >{PE_HIGH_BOUND}x = Above):")
    for f in ("Above Avg","Within Avg","Below Avg","N/A"):
        n   = int(flags.get(f, 0))
        bar = "█" * int(n/max(total,1)*40)
        print(f"    {f:<14}  {n:>3} co.  ({n/max(total,1)*100:5.1f}%)  {bar}")
    print(f"\n  S&P 500 INDEX  (Market-Cap Weighted):")
    print(f"    Wtd Avg NTM P/E : {sp_ntm:.1f}x  (vs {HIST_AVG_PE}x hist avg → {(sp_ntm/HIST_AVG_PE-1)*100:+.1f}%)")
    print(f"    Wtd Avg LTM P/E : {sp_ltm:.1f}x  (vs {HIST_AVG_PE}x hist avg → {(sp_ltm/HIST_AVG_PE-1)*100:+.1f}%)")
    print(f"    Total Mkt Cap   : ${total_mc/1_000_000:,.2f}T")
    print(f"\n  TOP 5 MOST EXPENSIVE SUB-SECTORS:")
    for i, (_, r) in enumerate(vsub.head(5).iterrows(), 1):
        print(f"    {i}. {r.iloc[0]:<44} {r['Weighted Avg NTM P/E']:.1f}x")
    print(f"\n  TOP 5 CHEAPEST SUB-SECTORS:")
    for i, (_, r) in enumerate(vsub.nsmallest(5,'Weighted Avg NTM P/E').iterrows(), 1):
        print(f"    {i}. {r.iloc[0]:<44} {r['Weighted Avg NTM P/E']:.1f}x")
    print("\n" + "═"*72)


In [17]:
from pathlib import Path
from datetime import datetime

# JUPYTER ENTRY POINT (no CLI)
AS_OF_STR   = "2025-04-11"  # or None for today
DEMO_MODE   = False          # True = use built-in sample; False = live Finviz
MAX_ROWS    = None          # optional cap for live scrape

as_of = datetime.strptime(AS_OF_STR, "%Y-%m-%d") if AS_OF_STR else datetime.today()
date_str = as_of.strftime("%Y-%m-%d")
OUTPUT_PATH = Path(f"SP500_Valuation_Analysis_{date_str}.xlsx")

log.info(f"Report as-of date : {as_of:%B %d, %Y}")

if DEMO_MODE:
    log.info("DEMO MODE — using built-in sample.")
    raw = build_demo_df()
    df  = standardise(raw)
else:
    log.info("LIVE MODE — scraping Finviz …")
    raw = FinvizScraper(delay=1.2).fetch_sp500(max_rows=MAX_ROWS)
    df  = standardise(raw)

log.info(f"Clean dataset : {len(df)} companies.")

sub_df = build_summary(df, "GICS Sub-Industry")
sec_df = build_summary(df, "GICS Sector")

wb = build_workbook(df, sub_df, sec_df, as_of)
wb.save(str(OUTPUT_PATH))
print_summary(df, sub_df, as_of)
OUTPUT_PATH


20:27:18  INFO     Report as-of date : April 11, 2025
20:27:18  INFO     LIVE MODE — scraping Finviz …
20:27:18  INFO     Connecting to Finviz S&P 500 screener …
20:27:18  WARNING  Could not read total count — will paginate until empty page.
20:27:18  INFO     Finviz reports 600 S&P 500 constituents.
20:27:18  INFO       Page r=1 → 20 rows
20:27:20  INFO       Fetching rows 21–40 …
20:27:20  INFO       Running total: 40 rows
20:27:22  INFO       Fetching rows 41–60 …
20:27:22  INFO       Running total: 60 rows
20:27:24  INFO       Fetching rows 61–80 …
20:27:24  INFO       Running total: 80 rows
20:27:26  INFO       Fetching rows 81–100 …
20:27:26  INFO       Running total: 100 rows
20:27:27  INFO       Fetching rows 101–120 …
20:27:27  INFO       Running total: 120 rows
20:27:29  INFO       Fetching rows 121–140 …
20:27:29  INFO       Running total: 140 rows
20:27:30  INFO       Fetching rows 141–160 …
20:27:31  INFO       Running total: 160 rows
20:27:32  INFO       Fetching rows 161


════════════════════════════════════════════════════════════════════════
   S&P 500 VALUATION ANALYSIS — CONSOLE SUMMARY
   April 11, 2025  |  ECM Research
════════════════════════════════════════════════════════════════════════

  DATA COVERAGE (Finviz S&P 500 screener):
    Companies captured              : 503
    Missing NTM P/E (no consensus)  : 6  (1.2%)
    Missing LTM P/E (neg. earnings) : 28  (5.6%)

  VALUATION FLAGS  (thresholds: <12.72x = Below, 12.72–17.72x = Within, >17.72x = Above):
    Above Avg       228 co.  ( 45.3%)  ██████████████████
    Within Avg      124 co.  ( 24.7%)  █████████
    Below Avg       145 co.  ( 28.8%)  ███████████
    N/A               6 co.  (  1.2%)  

  S&P 500 INDEX  (Market-Cap Weighted):
    Wtd Avg NTM P/E : 26.9x  (vs 15.22x hist avg → +76.7%)
    Wtd Avg LTM P/E : 44.8x  (vs 15.22x hist avg → +194.3%)
    Total Mkt Cap   : $72.05T

  TOP 5 MOST EXPENSIVE SUB-SECTORS:
    1. Auto Manufacturers                           156.3x
    2. REIT 

WindowsPath('SP500_Valuation_Analysis_2025-04-11.xlsx')